# DriftGuard — Baseline Models

## About This Notebook

In the previous notebook, I validated the Electricity dataset, explored its structure, and prepared a model-ready representation.

In this notebook, I will establish baseline classification performance using Gaussian Naive Bayes and K-Nearest Neighbors (KNN).

These baseline models will provide a reference point for the later streaming and adaptive experiments.

### What I Will Do

I will:

- Prepare the feature matrix and target
- Create a baseline train-test split
- Train Gaussian Naive Bayes
- Train KNN
- Evaluate both models using Accuracy, F1-score, and Recall
- Compare their performance
- Save the baseline results for later experiments

> **Important:** These are baseline experiments. Drift detection and adaptive learning will be introduced in later notebooks.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score

In [4]:
from pathlib import Path
import pandas as pd
from scipy.io import arff
data_path = Path("../data/raw/elecNormNew.arff")
data, metadata = arff.loadarff(data_path)
df = pd.DataFrame(data)
model_data = df.copy()
model_data["day"] = model_data["day"].str.decode("utf-8")
model_data["class"] = model_data["class"].str.decode("utf-8")
model_data = pd.get_dummies(model_data, columns=["day"], dtype=int)
model_data["class"] = model_data["class"].map({"DOWN": 0, "UP": 1})
print("Model-ready shape:", model_data.shape)

Model-ready shape: (45312, 15)


## Preparing the Features and Target

### Why

I now need to separate the predictor variables from the target variable.

The models should receive only information that is available as input, while `class` must remain the value they are trying to predict.

I will also preserve the existing row order when creating the feature matrix and target.

In [5]:
# Separate the predictor variables from the target
X = model_data.drop(columns=["class"])

# Store the target variable separately
y = model_data["class"]

# Check the dimensions of the features and target
print("Features:", X.shape)
print("Target:", y.shape)

Features: (45312, 14)
Target: (45312,)


## Chronological Train-Test Split

### Why

I need a baseline that reflects how the model would encounter data in a real stream.

A random train-test split could place observations from later points in the stream into the training data and earlier observations into the test data. This would weaken the temporal nature of the problem and could introduce information leakage.

I will therefore preserve the original order and use the earlier observations for training and the later observations for testing.

### Decision

I will use an 80% training portion and a 20% testing portion for this baseline experiment.

This split is only for establishing baseline performance. It is not the batch size that I will use later in the streaming simulation.

In [6]:
# Calculate the number of observations to use for baseline training
train_size = int(len(X) * 0.80)

# Select the earlier observations for training
X_train = X.iloc[:train_size]
y_train = y.iloc[:train_size]

# Select the later observations for testing
X_test = X.iloc[train_size:]
y_test = y.iloc[train_size:]

# Display the sizes of the chronological training and testing sets
print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

Training observations: 36249
Testing observations: 9063


## Baseline 1 — Gaussian Naive Bayes

### Why

Gaussian Naive Bayes is one of the two classifiers I selected for DriftGuard.

I first need to establish how a standard batch-trained version performs before introducing incremental learning and Welford-based updates.

This result will become the reference point for the adaptive Gaussian Naive Bayes model later in the project.

In [7]:
# Import the standard Gaussian Naive Bayes classifier
from sklearn.naive_bayes import GaussianNB

# Create the Gaussian Naive Bayes baseline model
nb_model = GaussianNB()

# Train the model using the earlier portion of the data stream
nb_model.fit(X_train, y_train)

# Generate predictions for the later portion of the data stream
nb_predictions = nb_model.predict(X_test)

## Baseline 2 — K-Nearest Neighbors

### Why

KNN is the second classifier I selected for DriftGuard.

KNN predicts a new observation by comparing it with stored training observations. This makes it fundamentally different from Gaussian Naive Bayes and gives me a useful second model for the adaptive comparison.

I will first establish the performance of standard KNN before introducing the sliding reference window later in the project.

In [8]:
# Import the K-Nearest Neighbors classifier
from sklearn.neighbors import KNeighborsClassifier

# Create the KNN baseline model using the standard initial configuration
knn_model = KNeighborsClassifier()

# Store the chronological training observations in the KNN model
knn_model.fit(X_train, y_train)

# Generate predictions for the later portion of the data stream
knn_predictions = knn_model.predict(X_test)

## Baseline Evaluation

### Why

I now need to measure how well both baseline models classify the unseen later portion of the stream.

I will use Accuracy, F1-score, and Recall. Using multiple metrics is important because the target classes are not perfectly balanced, so accuracy alone does not give a complete view of model performance.

I will keep these metrics consistent throughout the project so that the static and adaptive models can be compared fairly.

In [9]:
# Import the evaluation metrics used throughout the project
from sklearn.metrics import accuracy_score, f1_score, recall_score

# Calculate the evaluation metrics for Gaussian Naive Bayes
nb_accuracy = accuracy_score(y_test, nb_predictions)
nb_f1 = f1_score(y_test, nb_predictions)
nb_recall = recall_score(y_test, nb_predictions)

# Calculate the evaluation metrics for KNN
knn_accuracy = accuracy_score(y_test, knn_predictions)
knn_f1 = f1_score(y_test, knn_predictions)
knn_recall = recall_score(y_test, knn_predictions)

# Display the Gaussian Naive Bayes baseline performance
print("Gaussian Naive Bayes")
print("Accuracy:", nb_accuracy)
print("F1-score:", nb_f1)
print("Recall:", nb_recall)

# Display the KNN baseline performance
print("\nKNN")
print("Accuracy:", knn_accuracy)
print("F1-score:", knn_f1)
print("Recall:", knn_recall)

Gaussian Naive Bayes
Accuracy: 0.8064658501599912
F1-score: 0.7631650013502566
Recall: 0.6911225238444607

KNN
Accuracy: 0.6208760895950568
F1-score: 0.5622929936305733
Recall: 0.5397407679139153


## Baseline Model Comparison

### Why

I want to store the baseline results in a structured format instead of relying only on notebook output.

This allows me to reuse the results later when comparing static models with incremental and adaptive models.

In [10]:
# Store the evaluation results for both baseline models
baseline_results = pd.DataFrame({
    "Model": ["Gaussian Naive Bayes", "KNN"],
    "Accuracy": [nb_accuracy, knn_accuracy],
    "F1-score": [nb_f1, knn_f1],
    "Recall": [nb_recall, knn_recall]
})

# Display the baseline results for comparison
baseline_results

,Model,Accuracy,F1-score,Recall
0,Gaussian Naive Bayes,0.806466,0.763165,0.691123
1,KNN,0.620876,0.562293,0.539741


In [11]:
# Save the baseline results for comparison with later experiments
baseline_results.to_csv("../reports/results/baseline_results.csv", index=False)

# Key Findings and Decisions

I established chronological baseline performance for Gaussian Naive Bayes and KNN using the Electricity dataset.

### Baseline Setup

I used the earlier 80% of the ordered observations for training and the later 20% for testing. I did not shuffle the data because preserving the original sequence is essential for the streaming problem.

### Model Performance

Both models were evaluated using Accuracy, F1-score, and Recall. These metrics will remain consistent throughout the project so that I can make a fair comparison between the baseline, incremental, and adaptive approaches.

The baseline results have also been saved as a CSV file so that they can be reused during the later evaluation stages.

### Decision

These static baseline models will serve as the reference point for the rest of DriftGuard.

I will now move from conventional batch learning to a simulated streaming environment. The goal will be to determine how the models behave when observations arrive sequentially rather than being available all at once.
